# Lab 1 — ReAct Loop (§19.2.1)

In this lab we'll write a **ReAct agent** from scratch, without a framework, to see how the harness works.

**What you'll learn**
1. Call an LLM through an API and read the result
2. Build tools for the LLM to call
3. Write a ReAct loop with all 4 components from the textbook

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.2.1 — CC BY-SA 4.0

## 0. Set up before you start

| What you need | Details |
|---|---|
| Python 3.10 or later | or use Google Colab, which already has it |
| OpenRouter API key | sign up at openrouter.ai → Keys → Create Key |
| Packages | `openai`, `python-dotenv` (installed in the next cell) |

**Add your API key**
- **Colab:** 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file in the same folder as the notebook and add the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key directly in your code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

## 1. Calling an LLM through OpenRouter

We use **Gemma 4 26B** (open weights) through OpenRouter, which has an *OpenAI-compatible* API

In [ ]:
import os, json, time
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

In [ ]:
def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

In [ ]:
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"provider": {"require_parameters": True}, "reasoning": {"effort": "medium"}}

In [ ]:
def ask_llm(messages, tools=None):
    kwargs = dict(model=MODEL, messages=messages, extra_body=EXTRA)
    if tools:
        kwargs["tools"] = tools
    resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message

In [ ]:
msg = ask_llm([{"role": "user", "content": "Explain what an AI agent is in one sentence."}])
print("💭", (getattr(msg, "reasoning", None) or "")[:300])
print("💬", msg.content)

## 2. Building Tools

Tool = **Python function** + **JSON schema** that tells the model what the tool does and which arguments to send
The textbook stresses *"Provide good tools"* (§19.3): a clear description, a strict schema

In [ ]:
TOOLS = {}

def tool(description, properties):
    def wrap(fn):
        TOOLS[fn.__name__] = {"fn": fn, "schema": {"type": "function", "function": {
            "name": fn.__name__,
            "description": description,
            "parameters": {"type": "object", "properties": properties,
                           "required": list(properties), "additionalProperties": False},
        }}}
        return fn
    return wrap

def schemas(names):
    return [TOOLS[n]["schema"] for n in names]

In [ ]:
PAPERS = {  # simulated data for teaching
    "P1": {"title": "Crossformer", "year": 2023, "method": "two-stage attention across dimensions"},
    "P2": {"title": "TimesFM", "year": 2024, "method": "decoder-only foundation model"},
    "P3": {"title": "PatchTST", "year": 2023, "method": "patching + channel independence"},
    "P4": {"title": "iTransformer", "year": 2024, "method": "attention over variates, not time steps"},
}

@tool("Search papers on a topic. Returns a JSON list of {id, title, year}.",
      {"query": {"type": "string"}})
def search_papers(query):
    return json.dumps([{"id": k, "title": p["title"], "year": p["year"]} for k, p in PAPERS.items()])

@tool("Read papers by id (e.g. 'P1') and return their key methods.",
      {"paper_ids": {"type": "array", "items": {"type": "string"}}})
def read_papers(paper_ids):
    unknown = [i for i in paper_ids if i not in PAPERS]
    if unknown:
        return f"Error: unknown ids {unknown}. Use ids from search_papers, e.g. 'P1'."
    return json.dumps([{"title": PAPERS[i]["title"], "method": PAPERS[i]["method"]} for i in paper_ids])

In [ ]:
print(search_papers("time series"))
print(json.dumps(schemas(["search_papers"]), indent=2))

## 3. ReAct loop

$$\text{Thought}_t \rightarrow \text{Action}_t \rightarrow \text{Observation}_t \rightarrow \text{Thought}_{t+1}$$

The harness needs 4 things: ① scratchpad ② tool parsing ③ max iterations ④ termination

In [ ]:
def run_tool(call):
    try:
        args = json.loads(call.function.arguments)
        out = TOOLS[call.function.name]["fn"](**args)
    except Exception as e:
        out = f"Error: {e}"
    return {"role": "tool", "tool_call_id": call.id, "content": str(out)}

In [ ]:
def react_agent(task, tool_names, max_iterations=10):
    messages = [{"role": "user", "content": task}]
    for i in range(1, max_iterations + 1):           # ③ max iterations
        msg = ask_llm(messages, tools=schemas(tool_names))
        thought = getattr(msg, "reasoning", None)    # ① scratchpad
        if thought:
            print(f"💭 [{i}]", thought[:200])
        calls = msg.tool_calls or []                 # ② tool parsing
        if not calls:                                # ④ termination
            return msg.content
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [c.model_dump() for c in calls]})
        for c in calls:
            print(f"🔧 [{i}] {c.function.name}({c.function.arguments})")
        with ThreadPoolExecutor() as pool:
            results = list(pool.map(run_tool, calls))
        for r in results:
            print(f"👀 [{i}]", r["content"][:150])
        messages.extend(results)
    return "⚠️ Stopped: reached max_iterations"

In [ ]:
answer = react_agent("Find transformer models for time-series forecasting, read them, "
                     "and explain in 3 bullet points how they differ.",
                     ["search_papers", "read_papers"])
print("\n✅", answer)

In [ ]:
print(react_agent("Find transformer models for time-series forecasting, read them, then compare them.",
                  ["search_papers", "read_papers"], max_iterations=1))

## Summary

| Component | In the code |
|---|---|
| ① Scratchpad | `getattr(msg, "reasoning")` — logged but not shown to the user |
| ② Tool parsing | `msg.tool_calls` |
| ③ Max iterations | `for i in range(1, max_iterations + 1)` |
| ④ Termination | `if not calls: return msg.content` |

**Try next:** add a `calculator` tool and ask a question that needs arithmetic